In [1]:
import pandas as pd

df = pd.read_csv('data_files/Heart_disease.csv')
df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [2]:
df.shape

(4240, 16)

In [3]:
df.isnull().sum()

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [4]:
# Drop Eduaction column 
df.drop('education',axis = 1,inplace=True)

## Fill Missing Values


In [5]:
df.isnull().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [6]:
# Define the binary columns 
# To fill missing values (NaNs) in binary columns using the most frequent value (mode) from each column.

# Because for binary features, the most logical and safe way to fill missing values is to use the most frequent value (mode).

# For example:
# If 70% of people in the data are non-smokers (0) and 30% are smokers (1), then if someone's value is missing, it makes sense to assume they are more likely to be a non-smoker.



bin_cols = ['male','currentSmoker','prevalentStroke','prevalentHyp','diabetes']


for col in bin_cols:
    mode_values = df[col].mode()[0]
    df[col].fillna(mode_values, inplace=True)

C:\Users\gaure\AppData\Local\Temp\ipykernel_21704\3911885028.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mode_values, inplace=True)
C:\Users\gaure\AppData\Local\Temp\ipykernel_21704\3911885028.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example,

In [7]:
df.isnull().sum()

male                 0
age                  0
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

In [8]:
# Import necessary Libaries

import numpy as np

# Fill missing values for numeric features

numeric_cols = ['cigsPerDay','BPMeds','totChol','BMI','heartRate','glucose']


for col in numeric_cols:
    median_Val = df[col].median()
    df[col].fillna(median_Val, inplace=True)
    

C:\Users\gaure\AppData\Local\Temp\ipykernel_21704\3018032783.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_Val, inplace=True)


In [9]:
df.isnull().sum()

male               0
age                0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64

## Balance Dataset

In [10]:
# This is inbalanced data
df['TenYearCHD'].value_counts()

TenYearCHD
0    3596
1     644
Name: count, dtype: int64

In [11]:
from sklearn.utils import resample


# df_majority: All rows where TenYearCHD == 0 (no heart disease)
# df_minority: All rows where TenYearCHD == 1 (at risk of heart disease)



df_majority = df[df['TenYearCHD'] == 0]
df_minority = df[df['TenYearCHD'] == 1]


# Upsample the Minority Class

df_minority_upsampled = resample(df_minority,
                                 replace=True,
                                 n_samples=len(df_majority),
                                 random_state=42)

# replace=True: You allow duplicate rows to be selected (with replacement).
# n_samples=len(df_majority): You create the same number of rows as the majority class — e.g., 2500 rows.
# random_state=42: For reproducibility — so every time you run it, results are the same.




# Combine the Two to Get a Balanced Dataset.
df_balanced = pd.concat([df_majority,df_minority_upsampled])



In [12]:
df_balanced['TenYearCHD'].value_counts()

TenYearCHD
0    3596
1    3596
Name: count, dtype: int64

## Train Test Split and Feature Scalling


In [13]:
from sklearn.model_selection import train_test_split

# Separte Features (X) and Target variable(y)

X = df_balanced.drop(columns=['TenYearCHD'])
y = df_balanced['TenYearCHD']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)


# X_train and y_train is used for train the model
# X_test  = testing
# y_test = prediction

## Scaling (StandardScale)


In [14]:
from sklearn.preprocessing import StandardScaler


# what is Scaling ?
# Scaling means changing the range of your numerical data to a standard range or format.
# Think of it like resizing all features so they are on the same level, which makes it easier for machine learning algorithms to understand and compare them.



scaler = StandardScaler()

# Fit scaler to training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##

In [15]:
X_train_scaled

array([[-0.94172615,  1.4582083 , -1.02624378, ...,  0.840088  ,
        -0.50731448, -0.34072887],
       [-0.94172615, -1.66694628,  0.97442734, ..., -1.050363  ,
        -0.0898974 ,  0.25133934],
       [-0.94172615,  1.34246184,  0.97442734, ...,  0.42574258,
        -1.34214864, -0.34072887],
       ...,
       [ 1.06187982,  1.92119417, -1.02624378, ..., -0.6972277 ,
        -1.34214864, -0.18492145],
       [ 1.06187982,  1.68970124,  0.97442734, ...,  0.59289329,
        -0.0898974 ,  4.08420194],
       [ 1.06187982, -0.74097455,  0.97442734, ...,  1.0660946 ,
         0.32751968,  1.84057505]])

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()

rf_model.fit(X_train_scaled,y_train)

y_pred = rf_model.predict(X_test_scaled)

In [17]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy_score(y_test,y_pred)

0.9728978457261988

In [18]:
confusion_matrix(y_test,y_pred)

array([[701,  34],
       [  5, 699]], dtype=int64)

In [19]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       735
           1       0.95      0.99      0.97       704

    accuracy                           0.97      1439
   macro avg       0.97      0.97      0.97      1439
weighted avg       0.97      0.97      0.97      1439



## Training 10 Models With Different Metrics


In [20]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix

In [21]:
# Define a list of classifiers
classifiers = [
    RandomForestClassifier(),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    LogisticRegression(),
    SVC(),
    KNeighborsClassifier(),
    DecisionTreeClassifier(),
    GaussianNB(),
    XGBClassifier()
]

# Create a dictionary to store the results
results = {}


# Train and evaluate each classifier
for clf in classifiers:
    clf_name = clf.__class__.__name__
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{clf_name} Accuracy: {accuracy}")
    
    # Classification report
    print(f"Classification Report for {clf_name}:")
    print(classification_report(y_test, y_pred))
    
    # Confusion matrix
    print(f"Confusion Matrix for {clf_name}:")
    print(confusion_matrix(y_test, y_pred))
    print("="*50)

RandomForestClassifier Accuracy: 0.9666435024322446
Classification Report for RandomForestClassifier:
              precision    recall  f1-score   support

           0       0.99      0.94      0.97       735
           1       0.94      0.99      0.97       704

    accuracy                           0.97      1439
   macro avg       0.97      0.97      0.97      1439
weighted avg       0.97      0.97      0.97      1439

Confusion Matrix for RandomForestClassifier:
[[692  43]
 [  5 699]]
AdaBoostClassifier Accuracy: 0.6719944405837387
Classification Report for AdaBoostClassifier:
              precision    recall  f1-score   support

           0       0.69      0.66      0.67       735
           1       0.66      0.68      0.67       704

    accuracy                           0.67      1439
   macro avg       0.67      0.67      0.67      1439
weighted avg       0.67      0.67      0.67      1439

Confusion Matrix for AdaBoostClassifier:
[[486 249]
 [223 481]]
GradientBoostingCl

## Show Each Model Results


In [22]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define a list of classifiers
classifiers = [
    RandomForestClassifier(),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    LogisticRegression(),
    SVC(),
    KNeighborsClassifier(),
    DecisionTreeClassifier(),
    GaussianNB(),
    XGBClassifier()
]

# Create an empty list to collect results
results_list = []

# Train and evaluate each classifier
for clf in classifiers:
    clf_name = clf.__class__.__name__
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1_score = report['weighted avg']['f1-score']
    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']
    
    # Add results to list
    results_list.append({
        'Model': clf_name,
        'Accuracy': accuracy,
        'F1-Score': f1_score,
        'Precision': precision,
        'Recall': recall
    })

# Convert list of dicts to DataFrame
results_df = pd.DataFrame(results_list)
results_df

,Model,Accuracy,F1-Score,Precision,Recall
0,RandomForestClassifier,0.972898,0.972899,0.973690,0.972898
1,AdaBoostClassifier,0.671994,0.672015,0.672474,0.671994
2,GradientBoostingClassifier,0.728978,0.728702,0.731320,0.728978
3,LogisticRegression,0.658791,0.658830,0.659053,0.658791
4,SVC,0.683113,0.683126,0.683656,0.683113
5,KNeighborsClassifier,0.787352,0.783833,0.812481,0.787352
6,DecisionTreeClassifier,0.917999,0.917638,0.928593,0.917999
7,GaussianNB,0.583044,0.530092,0.635597,0.583044
8,XGBClassifier,0.906185,0.905977,0.912148,0.906185


## Best Model


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Instantiate the RandomForestClassifier
rf_classifier = RandomForestClassifier()

# Train the RandomForestClassifier
rf_classifier.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred_rf = rf_classifier.predict(X_test_scaled)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest Classifier Accuracy:", accuracy_rf)

# Classification report
print("Classification Report for Random Forest Classifier:")
print(classification_report(y_test, y_pred_rf))

# Confusion matrix
print("Confusion Matrix for Random Forest Classifier:")
print(confusion_matrix(y_test, y_pred_rf))


Random Forest Classifier Accuracy: 0.9742876997915219
Classification Report for Random Forest Classifier:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97       735
           1       0.95      1.00      0.97       704

    accuracy                           0.97      1439
   macro avg       0.97      0.97      0.97      1439
weighted avg       0.98      0.97      0.97      1439

Confusion Matrix for Random Forest Classifier:
[[701  34]
 [  3 701]]


## Single prediction


🎯 Goal of This Code
To test your trained Random Forest model (rf_classifier) on a single test sample (the 11th one), and print:

What the model predicted (predicted class)

What the actual label is (actual class)

In [24]:
# test 1:
print("predcted class ",rf_classifier.predict(X_test_scaled[10].reshape(1,-1))[0])
# This selects the 11th row (Python is 0-indexed) from your scaled test features.


print("Actual Class: ",y_test.iloc[10])


predcted class  0
Actual Class:  0


In [25]:
print("Predicated Class: ",rf_classifier.predict(X_test_scaled[200].reshape(1,-1))[0])
print("Actual Class: ",y_test.iloc[200])


Predicated Class:  1
Actual Class:  1


In [26]:
print("predcted class ",rf_classifier.predict(X_test_scaled[110].reshape(1,-1))[0])
print("actual class ", y_test.iloc[110])

predcted class  1
actual class  1


## Save Models

In [27]:
import pickle 

pickle.dump(rf_classifier,open('rf_classifier.pkl','wb'))
pickle.dump(scaler,open('scaler.pkl','wb'))

## Load models to test1


In [28]:
import pickle

# load the RandomForestClassifier Model:

# "rf_classifier.pkl": This is the file name where you saved your trained RandomForestClassifier.
# 'rb': Means you're opening the file in read-binary mode.
# pickle.load(file): Loads the model into Python.
# Now, rf_classifier is ready to use for prediction!



with open("rf_classifier.pkl",'rb') as file:
    rf_classifier = pickle.load(file)

# Load the scaler
with open("scaler.pkl",'rb') as file:
    scaler = pickle.load(file)

In [ ]:
# 🔍 Purpose of the Function
# This function is used to:

# Take raw patient input (like age, smoking habit, glucose level, etc.)
# Convert it into a format the machine learning model understands
# Scale the data using the same StandardScaler used during training
# Make a prediction using the trained model (e.g., RandomForest)
# Return the predicted result:
     # 1 → Patient has heart disease
     # 0 → Patient does not have heart disease



def predict(model, scaler, male, age, currentSmoker, cigsPerDay, BPMeds, prevalentStroke, prevalentHyp, diabetes, totChol, sysBP, diaBP, BMI, heartRate, glucose):
    


    # Encode categorical variables
    # Converts text input like "male" or "yes" into 1, and "female" or "no" into 0.
    # This is important because machine learning models only understand numeric value

    male_encoded = 1 if male.lower() == "male" else 0
    currentSmoker_encoded = 1 if currentSmoker.lower() == "yes" else 0
    BPMeds_encoded = 1 if BPMeds.lower() == "yes" else 0
    prevalentStroke_encoded = 1 if prevalentStroke.lower() == "yes" else 0
    prevalentHyp_encoded = 1 if prevalentHyp.lower() == "yes" else 0
    diabetes_encoded = 1 if diabetes.lower() == "yes" else 0



    # Prepare features array
    # Puts all input features into a 2D NumPy array (shape: (1, 15)).
    # Format is critical—models expect input to be 2D (like a matrix, even for single predictions).
    features = np.array([[male_encoded, age, currentSmoker_encoded, cigsPerDay,
                          BPMeds_encoded, prevalentStroke_encoded, prevalentHyp_encoded,
                          diabetes_encoded, totChol, sysBP, diaBP, BMI, heartRate, glucose]])
 
    
    # Apply Feature Scalling
    # Applies standard scaling to the input using the same scaler used during training.
    # Ensures that each feature has the same scale (e.g., mean = 0, std = 1), which improves model performance.

    # Scale the input features using the same scaler used during training
    scaled_feature = scaler.transform(features)
    
    # Predict using the trained model
    result = model.predict(scaled_feature)
    
    # Return the predicted value (0 or 1)
    return result[0]
    




In [30]:
# Test 1
male = "female"
age = 56.00
currentSmoker = "yes"
cigsPerDay = 3.00
BPMeds = "no"
prevalentStroke = "no"
prevalentHyp = "yes"
diabetes = 'no'
totChol = 285.00
sysBP = 145.00
diaBP = 100.00
BMI = 30.14
heartRate = 80.00
glucose = 86.00

result = predict(rf_classifier, scaler, male, age, currentSmoker, cigsPerDay, BPMeds,
                 prevalentStroke, prevalentHyp, diabetes, totChol, sysBP, diaBP, BMI, heartRate, glucose)

if result == 1:
    print("✅ The Patient has Heart Disease")
else:
    print("✅ The Patient has No Heart Disease")


✅ The Patient has No Heart Disease


C:\Users\gaure\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [31]:
# test 2
male = 'female'
age = 63.0
currentSmoker = 'yes'
cigsPerDay = 3.0
BPMeds = 'no'
prevalentStroke = 'no'
prevalentHyp = 'yes'
diabetes = 'no'
totChol = 267.0
sysBP = 156.5
diaBP = 92.5
BMI = 27.1
heartRate = 60.0
glucose = 79.0
result = 1.0



result = predict(rf_classifier, scaler, male, age, currentSmoker, cigsPerDay, BPMeds, prevalentStroke, prevalentHyp, diabetes, totChol, sysBP, diaBP, BMI, heartRate, glucose)


if result == 1:
    print("The Patient has Heart Diseas")
else: 
    print("The Patiennt has No Heart Deseas")

The Patient has Heart Diseas


C:\Users\gaure\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
